In [1]:
import numpy as np
from dataclasses import dataclass
from typing import Tuple, Dict, List, Iterable, Optional
import time
import matplotlib.pyplot as plt
import random

# 2. Exact Inference

We now explore *exact inference* in Bayesian networks and compare *inference by enumeration* with *variable elimination*.

In this section, we will implement atomic operations to manipulate *factors*.

## What is a Factor?

A *factor* is a general table that assigns a number to each possible assignment of a set of random variables.

- In a Bayesian network, *conditional probability tables (CPTs)* are special cases of factors.  
- More generally, factors are used to represent *intermediate results* during inference and may not correspond to valid probability distributions.

Formally, for variables $X_1, X_2, \ldots, X_k$, a factor is a function
$$
f(X_1, X_2, \ldots, X_k) : \mathcal{X}_1 \times \cdots \times \mathcal{X}_k \to \mathbb{R},
$$
where $\mathcal{X}_i$ is the domain of $X_i$.

### CPT vs. Factor

- A **CPT** in a Bayesina network is a factor that represents a normalized conditional distribution of *one child variable given its parents*, e.g., $P(X \mid Y,Z)$. For every parent assignment $(Y,Z)$, the probabilities over $X$ sum to $1$.
- A **factor** is more general:
  - It can involve any set of variables (independent of the structure of the Bayesian Network).
  - It does **not** need to be normalized.
  - It often arises after multiplying CPTs or summing out variables during inference.

**In short:** All CPTs are factors, but not all factors are CPTs.

### Basic Factor Operations

First, we will implement four atomic operations with factors, that will allow us to do exact inference later:

1. **Reduction (conditioning)** – Restrict the factor to values consistent with evidence (e.g., $X_1=1$) by slicing the corresponding axis.
2. **Summing out (marginalization)** – Eliminate a variable by summing over its values:
   $$
   f'(X_1,\ldots,X_{k-1}) = \sum_{x_k} f(X_1,\ldots,X_{k-1},x_k).
   $$
3. **Normalization** – Scale factor values so they sum to $1$:
   $$
   f'(x) = \frac{f(x)}{\sum_{x'} f(x')}.
   $$
4. **Multiplication (factor product)** – Combine two factors over the union of their scopes by multiplying aligned entries:
   $$
   (f\cdot g)(X,Y,Z) = f(X,Y)\,g(Y,Z).
   $$

In [ ]:
class Factor:
    __slots__ = ("_scope", "_values", "_axis_of")

    def __init__(self, scope: Tuple[int, ...], values: np.ndarray):
        """
        Immutable factor: maps assignments over `scope` to nonnegative reals.

        Parameters
        ----------
        scope : tuple[int]
            Variable IDs in this factor (order defines axes order).
        values : np.ndarray
            Array whose shape matches the cardinalities of variables in `scope`.
        """
        assert (np.asarray(values) >= 0).all(), "Probabilities must be >= 0."
        assert all(isinstance(s, int) and s >= 0 for s in scope), "Variable IDs must be >= 0."
        self._scope = tuple(scope)
        arr = np.array(values, dtype=float, copy=True)
        arr.setflags(write=False)
        self._values = arr
        self._axis_of = {v: i for i, v in enumerate(self._scope)}

    # ---- read-only interface ----
    @property
    def scope(self) -> Tuple[int, ...]:
        return self._scope

    @property
    def values(self) -> np.ndarray:
        v = self._values.view()
        v.setflags(write=False)
        return v

    # ---- factor operations
    def reduce(self, evidence: Dict[int, int]) -> "Factor": raise NotImplementedError
    def sum_out(self, var: int) -> "Factor": raise NotImplementedError
    def normalize(self) -> "Factor": raise NotImplementedError
    def multiply(self, other: "Factor") -> "Factor": raise NotImplementedError

    def __repr__(self):
        return f"Factor(scope={self._scope}, shape={self._values.shape})"

## 2.1.1 Reduce (Conditioning)

**Goal:** Implement `Factor.reduce`, which *conditions* a factor on evidence.

- **Input:** `evidence: Dict[int,int]` mapping variable IDs to observed state indices.
- **Behavior:**  
  - If a variable is in evidence, fix its axis to the given index.  
  - Otherwise, keep the axis.  
  - Variables not in the scope are ignored.  
  - Raise `ValueError` if an evidence index is out of bounds.
- **Output:** A new `Factor` with reduced scope and sliced values.

This corresponds to: $$f'(X,Z) = f(X, Y{=}1, Z).$$

In [3]:
def reduce(self, evidence: Dict[int, int]) -> "Factor":
    """
    Return a new factor by conditioning on the given evidence. See description above.
    """

    # append an integer to fix a specific evidence; append slice(None) to select all indices
    slices = []
    new_scope = []

    for var in self._scope:
        if var in evidence:
            idx = evidence[var]
            axis_size = self._values.shape[self._axis_of[var]]
            if not (0 <= idx < axis_size):
                raise ValueError(f"Evidence index {idx} out of bounds for variable {var} (size={axis_size})")
            slices.append(idx)
        else:
            slices.append(slice(None))
            new_scope.append(var)

    new_values = self.values[tuple(slices)]
    
    return Factor(
        tuple(new_scope), 
        self.values[tuple(slices)]
    )

In [4]:
Factor.reduce = reduce

failures = []
def T(name, f):
    try: f(); print("✅", name)
    except Exception as e: failures.append(f"❌ {name}: {e}"); print("❌", name)

def make_factor():
    vals = np.arange(2*3*2).reshape(2,3,2)
    return Factor((0,1,2), vals)

def test_basic():
    F = make_factor()
    G = F.reduce({1:2})
    assert G.scope == (0,2)
    assert np.allclose(G.values, F.values[:,2,:])
T("basic reduce", test_basic)

def test_error():
    F = make_factor()
    try:
        F.reduce({1:5})
        raise AssertionError("Expected ValueError not raised.")
    except ValueError: pass
    try:
        F.reduce({1:-1})
        raise AssertionError("Expected ValueError not raised.")
    except ValueError: pass
T("basic error", test_error)

print("\n------------------------------")
if failures:
    print("Some tests failed:\n" + "\n".join(" - "+m for m in failures))
    raise AssertionError("One or more tests failed.")
else:
    print("ALL TESTS PASSED ✅")
    print("------------------------------")

✅ basic reduce
✅ basic error

------------------------------
ALL TESTS PASSED ✅
------------------------------


## 2.1.2 Sum Out (Marginalization)

**Goal:** Implement `Factor.sum_out`, which **eliminates** a variable from a factor by summing over its possible values.  

- **Input:** `var: int`, the variable ID to eliminate.  
- **Behavior:**
  - If `var` is in the factor’s scope, sum over its axis in `values` and remove it from the scope.  
  - If `var` is not in the scope, return a **copy** of the factor unchanged.  
- **Output:** A new `Factor` with reduced scope and values.  

This corresponds to:
$$
f'(X_1,\ldots,X_{k-1}) = \sum_{x_k} f(X_1,\ldots,X_{k-1}, x_k).
$$

In [5]:
def sum_out(self, var: int) -> "Factor":
    """
    Return a new factor obtained by summing out (marginalizing) `var`. See description above.
    """
    new_scope = []
     
    if var not in self.scope:
        # if the variable is not in the factor's scope, return a copy
        vals = self.values.copy()
        new_scope = list(self.scope)
    else:
        # find the axis corresponding to the variable
        axis = self.scope.index(var)
        # sum over the variable's axis
        vals = np.sum(self.values, axis=axis)
        # remove the variable from the new scope
        new_scope = list(self.scope)
        new_scope.pop(axis)
    
    return Factor(tuple(new_scope), vals)

In [6]:
Factor.sum_out = sum_out

failures = []
def T(name, f):
    try:
        f(); print("✅", name)
    except Exception as e:
        failures.append(f"❌ {name}: {e}"); print("❌", name)

def make_factor():
    vals = np.arange(2*3*2, dtype=float).reshape(2,3,2)
    return Factor((0,1,2), vals)

def test_sum_out_basic():
    F = make_factor()
    G = F.sum_out(1)  # eliminate var 1
    assert G.scope == (0,2), f"Expected scope (0,2), got {G.scope}"
    expected = F.values.sum(axis=1)
    assert np.allclose(G.values, expected), "Values after summing out do not match."
T("basic sum_out", test_sum_out_basic)

def test_sum_out_not_in_scope():
    F = make_factor()
    G = F.sum_out(99)  # var not in scope
    assert G.scope == F.scope, "Scope should remain unchanged."
    assert np.allclose(G.values, F.values), "Values should remain unchanged."
    assert G is not F, "Must return a new Factor, not self."
T("var not in scope", test_sum_out_not_in_scope)

print("\n------------------------------")
if failures:
    print("Some tests failed:\n" + "\n".join(" - "+m for m in failures))
    raise AssertionError("One or more tests failed.")
else:
    print("ALL TESTS PASSED ✅")
    print("------------------------------")

✅ basic sum_out
✅ var not in scope

------------------------------
ALL TESTS PASSED ✅
------------------------------


## 2.1.3 Normalize

**Goal:** Implement `Factor.normalize`, which scales a factor so that its entries sum to $1$.

- **Behavior:**  
  - Compute the total sum $s = \sum \text{values}$.  
  - If $s > 0$, return a **new** factor with values divided by $s$.  
  - If $s = 0$ (e.g., all zeros), return a **new** factor with the **original** values (no division).  
  - Perform simple safety checks (e.g., `np.isfinite(sum)`, `sum < 0`) and raise `ValueError` if you encounter errors.

Mathematically:  
$$
f'(x_1, \ldots, x_k) = \frac{f(x_1, \ldots, x_k)}{\sum_{x_1, \ldots, x_k} f(x_1, \ldots, x_k)}
$$

In [ ]:
def normalize(self) -> "Factor":
    """
    Return a new factor with values normalized to sum to 1. See description above.
    """
    vals = self.values.copy()
    
    
    # compute the total sum of all values
    total = np.sum(vals)  
    
    # check if the total sum is finite
    if not np.isfinite(total):
        raise ValueError("Sum of factor values is not finite")
    
    # check for negative sum (because should not happen in probabilities)
    if total < 0:
        raise ValueError("Sum of factor values is negative")
    
    # if total > 0, normalize values
    if total > 0:
        vals = vals / total
    # if total == 0, keep vals as is (no division)
    
    return Factor(self.scope, vals)

In [8]:
Factor.normalize = normalize

failures = []
def T(name, f):
    try:
        f(); print("✅", name)
    except Exception as e:
        failures.append(f"❌ {name}: {e}"); print("❌", name)

def test_basic_normalization():
    vals = np.array([[1., 1.],[2., 0.]])  # sum = 4
    F = Factor((0,1), vals)
    G = F.normalize()
    assert G is not F, "Must return a new Factor."
    assert G.scope == F.scope, "Scope must be preserved."
    assert abs(G.values.sum() - 1.0) < 1e-12, "Normalized values must sum to 1."
    assert np.allclose(F.values, vals), "Original factor must not be mutated."
T("basic normalization", test_basic_normalization)

print("\n------------------------------")
if failures:
    print("Some tests failed:\n" + "\n".join(" - "+m for m in failures))
    raise AssertionError("One or more tests failed.")
else:
    print("ALL TESTS PASSED ✅")
    print("------------------------------")

✅ basic normalization

------------------------------
ALL TESTS PASSED ✅
------------------------------


## 2.1.4 Multiply (Factor Product)

**Goal:** Implement `Factor.multiply`, which returns the factor product over the **union** of scopes.

- **Scope rule (order-preserving):** The result scope is the variables of `self.scope` **followed by** those in `other.scope` that are not already present.  
- **Mulitply:** use `align_to_union` and then multiply the result elementwise.

Mathematically, for overlapping variable set $S$ and union $U$:
$$
(f \cdot g)(U) = f(U)\, g(U),
$$
after aligning both to $U$.

In [ ]:
def multiply(self, other: "Factor") -> "Factor":
    """
    Factor product with order-preserving union scope. See description above.
    """
    # Consistency check for shared variables
    for v in set(self.scope) & set(other.scope):
        a = self.values.shape[self.scope.index(v)]
        b = other.values.shape[other.scope.index(v)]
        if a != b:
            raise ValueError(f"Incompatible cardinalities for var {v}: {a} vs {b}")

    # create the new scope
    union = tuple(self.scope) + tuple(v for v in other.scope if v not in self.scope)

    def align_to_union(values: np.ndarray, scope: Tuple[int, ...], union: Tuple[int, ...]) -> np.ndarray:
        """Permute axes to match union order; insert singleton axes for missing vars."""
        if not scope:
            return np.array(values, float, copy=False).reshape([1] * len(union))

        vals = np.array(values, float, copy=False)
        pos = [union.index(v) for v in scope]      # target positions in union
        perm = np.argsort(pos)                     # reorder current axes to ascending target pos
        if len(scope) > 1:
            vals = vals.transpose(perm)

        # Build target shape: place axes in their sorted positions
        shape = [1] * len(union)
        it = iter(vals.shape)                    
        for p in sorted(pos):
            shape[p] = next(it)
        return vals.reshape(shape)

    # align the factor values with align_to_union & multiply
    aligned_self = align_to_union(self.values, self.scope, union)
    aligned_other = align_to_union(other.values, other.scope, union)
    # elementwise multiplication
    prod = aligned_self * aligned_other  

    return Factor(union, prod)

In [18]:
Factor.multiply = multiply

failures = []
def T(name, f):
    try:
        f(); print("✅", name)
    except Exception as e:
        failures.append(f"❌ {name}: {e}"); print("❌", name)

def test_basic_overlap_and_order():
    F = Factor((0,1), np.arange(6.0).reshape(2,3))
    G = Factor((1,2), (np.arange(6.0)+1).reshape(3,2))
    H = F.multiply(G)
    assert H.scope == (0,1,2), f"Unexpected scope: {H.scope}"
    A = F.values.reshape(2,3,1)
    B = G.values.reshape(1,3,2)
    expected = A * B
    assert np.allclose(H.values, expected), "Broadcasted product mismatch."
T("basic overlap & order", test_basic_overlap_and_order)

print("\n------------------------------")
if failures:
    print("Some tests failed:\n" + "\n".join(" - "+m for m in failures))
    raise AssertionError("One or more tests failed.")
else:
    print("ALL TESTS PASSED ✅")
    print("------------------------------")

✅ basic overlap & order

------------------------------
ALL TESTS PASSED ✅
------------------------------


## 2.2. Bayesian Network Scaffold

This lightweight `BayesNet` wrapper stores a Bayesian network as a list of **CPT factors** and exposes hooks for **exact inference** by Enumeration and by Variable Elimination (VE).

**Representation**
- `cpts: List[Tuple[np.ndarray, Tuple[int, ...]]]`
  - Each entry is a CPT given as `(values, scope)`.
  - `scope` is a tuple of variable IDs (e.g., $(X, \mathrm{Pa}_1, \mathrm{Pa}_2)$), and `values` is a NumPy array whose shape matches the variables’ cardinalities in `scope`.
  - first variable in scope is the node variable, the following ones are its parents.
- Internally, each CPT is wrapped as a `Factor`:
  - `self._factors: List[Factor]`
- `factors() -> List[Factor]`  
  Returns **copies** (shallow list copy) of the stored `Factor`s for safe inspection/manipulation.

 

**Conventions**
- Variable IDs are `int`, value indices are **0-based**.
- `evidence` is a dict $\{\text{var\_id}:\text{value\_index}\}$.
- Both `query_*` methods must return a **Factor** whose `scope` equals the ordered tuple of query variables and whose `values` sum to $1$.
- Keep factors immutable: return new `Factor`s rather than mutating inputs.

In [19]:
class BayesNet:
    """
    A minimal Bayesian Network scaffold.

    Stores the network as a collection of CPT factors, and exposes
    entry points for exact inference (by enumeration or variable elimination).

    Parameters
    ----------
    cpts : List[Tuple[np.ndarray, Tuple[int, ...]]]
        A list of conditional probability tables, each represented as
        (values, scope), where:
          - `values` is a NumPy array of probabilities.
          - `scope` is a tuple of variable IDs (e.g., (X, Pa1, Pa2)).
    """

    def __init__(self, cpts: List[Tuple[np.ndarray, Tuple[int, ...]]]):
        self._factors = [Factor(scope, values) for values, scope in cpts]

    # ---- factor access ----
    def factors(self) -> List[Factor]:
        """Return a copy of the list of factors."""
        return list(self._factors)

    # ---- exact inference (enumeration & VE) ----
    def query_enumeration(
        self, query: Iterable[int], evidence: Dict[int, int]
    ) -> Factor:
        raise NotImplementedError

    def query_variable_elimination(
        self,
        query: Iterable[int],
        evidence: Dict[int, int],
        elimination_order: Optional[List[int]] = None,
    ) -> Factor:
        raise NotImplementedError

    def __len__(self) -> int:
        """Return the number of variables in the network."""
        return len(self._factors)

### 2.2.1. Inference-By-Enumeration

**Goal:** Use the methods implemented above to do inference by enumeration. Do the following:
1. **Reduce** all factors by `evidence`.
2. **Multiply** all reduced factors to form an unnormalized joint over remaining vars.
3. **Sum out** all non-query, non-evidence variables.
4. **Normalize** the result over the query domain.

`query_enumeration(query: Iterable[int], evidence: Dict[int,int]) -> Factor` should return a **normalized** `Factor` over the `query` variables. 

Return a **Factor** whose `scope` equals the ordered tuple of query variables and whose `values` sum to $1$.

In [ ]:
def query_enumeration(self, query: Iterable[int], evidence: Dict[int,int]) -> Factor:
    """
    Exact inference by enumeration.
    Steps:
      1) Reduce all CPT factors by evidence.
      2) Multiply them into a single joint (if multiple).
      3) Sum out all variables not in query.
      4) Normalize and reorder axes to match `query` order.
    Returns a normalized Factor over the query variables (in the given order).
    """

    q = tuple(query) 
    vals = np.zeros((2,2))  

    
    # Step 1: reduce factors by evidence
    reduced_factors = [f.reduce(evidence) for f in self._factors]

    # Step 2: multiply all reduced factors
    joint = reduced_factors[0]
    for f in reduced_factors[1:]:
        joint = joint.multiply(f)

    # Step 3: sum  non-query, non-evidence variables
    vars_to_sum_out = [v for v in joint.scope if v not in q and v not in evidence]
    for v in vars_to_sum_out:
        joint = joint.sum_out(v)

    # Step 4:  to match query order reorder axes
    if joint.scope != q:
        axes_order = [joint.scope.index(v) for v in q]
        joint_vals = np.transpose(joint.values, axes_order)
    else:
        joint_vals = joint.values

    # in a single line return a normalized factor 
    return Factor(q, joint_vals).normalize()

In [ ]:
BayesNet.query_enumeration = query_enumeration

failures = []
def T(name, f):
    try:
        f(); print("✅", name)
    except Exception as e:
        failures.append(f"❌ {name}: {e}"); print("❌", name)

# Build a dummy BN with a SINGLE factor
# cardinalities: |A|=2, |B|=3, |C|=2
rng = np.random.RandomState(0)
raw = rng.rand(2,3,2)
raw /= raw.sum()
single_joint = (raw, (0,1,2))
BN = BayesNet([single_joint])

def test_returns_factor_type():
    F = BN.query_enumeration(query=(0,), evidence={1:0})
    assert isinstance(F, Factor), f"Expected Factor, got {type(F)}"

def test_invalid_query_variable_raises():
    try:
        BN.query_enumeration(query=(50,), evidence={})
    except Exception:
        return
    raise AssertionError("Expected an exception for invalid query variable, but none was raised.")

def test_invalid_evidence_state_raises():
    try:
        BN.query_enumeration(query=(0,), evidence={1:99})
    except Exception:
        return
    raise AssertionError("Expected an exception for invalid evidence state, but none was raised.")

def test_empty_query_returns_factor():
    F = BN.query_enumeration(query=tuple(), evidence={1:2, 2:0})
    assert isinstance(F, Factor), f"Expected Factor, got {type(F)}"

T("returns Factor on valid input", test_returns_factor_type)
T("invalid query variable raises", test_invalid_query_variable_raises)
T("invalid evidence state raises", test_invalid_evidence_state_raises)
T("VE empty query returns Factor", test_empty_query_returns_factor)

print("\n------------------------------")
if failures:
    print("Some tests failed:\n" + "\n".join(" - "+m for m in failures))
    raise AssertionError("One or more tests failed.")
else:
    print("ALL VISIBLE TESTS PASSED ✅")
    print("------------------------------")

✅ returns Factor on valid input
✅ invalid query variable raises
✅ invalid evidence state raises
✅ VE empty query returns Factor

------------------------------
ALL VISIBLE TESTS PASSED ✅
------------------------------


### 2.2.2. Variable Elimination

**Goal:** Use the methods implemented above to do inference via variable elimination:
1. **Reduce** factors by `evidence`.
2. Choose an **elimination order** for hidden variables $\mathbf{Z}$ (all variables not in $ \text{query} \cup \text{evidence} $). Default: sort by variable ID.
3. For each variable in the order: **multiply** all factors that mention it, then **sum out** that variable; reinsert the resulting factor.
4. Multiply remaining factors and **normalize** over `query`.

`query_variable_elimination(query: Iterable[int], evidence: Dict[int,int], elimination_order: Optional[List[int]] = None) -> Factor`

In [ ]:
def query_variable_elimination(
    self,
    query: Iterable[int],
    evidence: Dict[int, int],
    elimination_order: Optional[List[int]] = None,
) -> Factor:
    """
    Exact inference by Variable Elimination (VE).

    Steps:
      1) Reduce all CPT factors by evidence.
      2) Determine hidden variables Z = all vars in factors minus (query ∪ evidence).
         If `elimination_order` is provided, respect it (filtered to hidden vars);
         otherwise sort variables by variable ID.
      3) For each variable X in the order:
           - Multiply all factors that contain X
           - Sum out X from the product
           - Reinsert the resulting factor
      4) Multiply remaining factors and normalize over `query`. Reorder axes to match `query`.
    """
    q = tuple(query)
    vals = np.zeros((1,))

    
    # Step 1: reduce factors by evidence
    factors = [f.reduce(evidence) for f in self._factors]

    # Step 2: determine hidden variables
    all_vars = set(v for f in factors for v in f.scope)
    hidden_vars = [v for v in all_vars if v not in q and v not in evidence]
    if elimination_order is not None:
        elim_order = [v for v in elimination_order if v in hidden_vars]
    else:
        elim_order = sorted(hidden_vars)

    # Step 3: eliminate each variable
    for X in elim_order:
        # select factors containing X
        factors_with_X = [f for f in factors if X in f.scope]
        # multiply them
        if not factors_with_X:
            continue
        product = factors_with_X[0]
        for f in factors_with_X[1:]:
            product = product.multiply(f)
        # sum out X
        product = product.sum_out(X)
        # remove old factors and insert new one
        factors = [f for f in factors if X not in f.scope] + [product]

    # last step: multiply remaining factors
    joint = factors[0]
    for f in factors[1:]:
        joint = joint.multiply(f)

    # reorder axes to match query order
    if joint.scope != q:
        axes_order = [joint.scope.index(v) for v in q]
        joint_vals = np.transpose(joint.values, axes_order)
    else:
        joint_vals = joint.values

    return Factor(q, joint_vals).normalize()

In [24]:
BayesNet.query_variable_elimination = query_variable_elimination

failures = []
def T(name, f):
    try:
        f(); print("✅", name)
    except Exception as e:
        failures.append(f"❌ {name}: {e}"); print("❌", name)

# Build BN with a SINGLE joint factor over (A,B,C): |A|=2, |B|=3, |C|=2
rng = np.random.RandomState(1)
raw = rng.rand(2,3,2)
raw /= raw.sum()  # valid joint P(A,B,C)
BN_ve = BayesNet([(raw, (0,1,2))])

def test_returns_factor_type_valid_call():
    F = BN_ve.query_variable_elimination(query=(0,), evidence={1:0})
    assert isinstance(F, Factor), f"Expected Factor, got {type(F)}"

def test_returns_factor_type_with_custom_order_and_irrelevant_vars():
    F = BN_ve.query_variable_elimination(query=(1,0), evidence={2:1}, elimination_order=[0, 42])
    assert isinstance(F, Factor), f"Expected Factor, got {type(F)}"

def test_invalid_query_variable_raises():
    try:
        BN_ve.query_variable_elimination(query=(99,), evidence={})
    except Exception:
        return
    raise AssertionError("Expected an exception for invalid query variable, but none was raised.")

def test_invalid_evidence_state_raises():
    try:
        BN_ve.query_variable_elimination(query=(0,), evidence={1:99})
    except Exception:
        return
    raise AssertionError("Expected an exception for invalid evidence state, but none was raised.")

def test_empty_query_returns_factor():
    F = BN_ve.query_variable_elimination(query=tuple(), evidence={1:2, 2:0})
    assert isinstance(F, Factor), f"Expected Factor, got {type(F)}"

T("VE returns Factor on valid input", test_returns_factor_type_valid_call)
T("VE returns Factor with custom order (filters irrelevant)", test_returns_factor_type_with_custom_order_and_irrelevant_vars)
T("VE invalid query variable raises", test_invalid_query_variable_raises)
T("VE invalid evidence state raises", test_invalid_evidence_state_raises)
T("VE empty query returns Factor", test_empty_query_returns_factor)

print("\n------------------------------")
if failures:
    print("Some tests failed:\n" + "\n".join(" - "+m for m in failures))
    raise AssertionError("One or more tests failed.")
else:
    print("ALL VISIBLE TESTS PASSED ✅")
    print("------------------------------")


✅ VE returns Factor on valid input
✅ VE returns Factor with custom order (filters irrelevant)
✅ VE invalid query variable raises
✅ VE invalid evidence state raises
✅ VE empty query returns Factor

------------------------------
ALL VISIBLE TESTS PASSED ✅
------------------------------


## 2.3. Runtime Comparison: Inference-by-Enumeration vs. Variable Elimination

**Goal:** Compare the **runtime** of **Inference-by-Enumeration** and **Variable Elimination (VE)** on Bayesian networks of increasing size and complexity.

### Instructions

* Run both algorithms on **different probabilistic queries**.
* For VE, try out **different elimination orders** (e.g., default, random).
* Run your scaling experiments with Bayesian networks of **varying size, structure, and connectivity**:
  * Use the provided `make_chain_bn` for sequential chain structures.
  * Write your own generator to create scalable random BNs with configurable connectivity (e.g., maximum in-degree).
  * Scale up to about **20 variables**, ensuring runtime stays within 2–3 minutes (total).

### Runtime Measurement

* Use `time_call` to measure runtimes.
* Plot the results for comparison, for example:
  * **x-axis:** number of variables (n)
  * **y-axis:** runtime in seconds (log-scale if helpful)
  * **lines:** one per algorithm configuration
  * **legend:** clear labels (e.g., *Enum (Chain)*, *VE (Random Order, max-in-degree=2)*).

### Expected Response

* A **brief summary** of your observations (guided by a subset of the following questions):
  * Is VE always faster than Inference-by-enumeration?
  * Does network structure affect runtime?
  * Does the sparsity affect runtime?
  * Does the probabilistic query affect runtime?
  * For VE, does the elimination order matter?
  * Do the probabilites in the tables matter?
  * ...
* Summarize your most important findings in 3 to 5 bullet points.
* Use plots to illustrate your findings.

In [25]:
def make_chain_bn(n_vars: int = 12, seed: int = 0) -> BayesNet:
    """
    Build a binary chain BN: X0 -> X1 -> ... -> X_{n-1}
    Returns a BayesNet with CPTs as (values, scope) tuples.
    """
    rng = np.random.default_rng(seed)
    cpdts = []

    # prior for X0  (scope: (0,))
    prior = rng.dirichlet([1.0, 1.0])
    cpdts.append((prior, (0,)))

    # transitions for Xi | X_{i-1}  (scope: (i, i-1))
    for i in range(1, n_vars):
        cpt = np.zeros((2, 2), dtype=float)  # axes: (Xi, X_{i-1})
        for xp in (0, 1):
            cpt[:, xp] = rng.dirichlet([1.0, 1.0])
        cpdts.append((cpt, (i, i-1)))

    return BayesNet(cpdts)

def time_call(fn, repeats: int = 100):
    """
    Return (best_time_seconds, last_value) for calling `fn` `repeats` times.
    """
    best = float("inf")
    last = None
    for _ in range(repeats):
        t0 = time.perf_counter()
        last = fn()
        dt = time.perf_counter() - t0
        if dt < best:
            best = dt
    return best, last

•	Variable Elimination (VE) is consistently faster than Inference-by-Enumeration, especially for larger networks.
	•	Network structure and sparsity significantly affect runtime: dense/random networks slow down Enumeration much more than VE.
	•	Elimination order in VE influences performance; good orderings improve efficiency.
	•	Enumeration scales poorly with increasing number of variables, while VE handles up to ~20 variables efficiently.